# 🧠 01 · 스파이킹 뉴런과 스파이크 인코딩

딥러닝은 이미지 인식이나 번역에서 사람에 필적하는 성능을 내지만, 그만한 성능을 학습하고 실행하는 데 많은 전력을 씁니다. 반면 사람의 뇌는 그에 못지않게 복잡한 일을 약 20W — 백열전구 하나 남짓한 전력 — 로 해냅니다. 이 큰 격차의 뿌리는 **뉴런이 정보를 주고받는 방식**에 있습니다.

인공신경망의 뉴런은 매 순간 연속적인 실수 값을 내보내지만, 생물학적 뉴런은 대부분의 시간을 침묵으로 보내다가 필요할 때만 짧은 전기 펄스인 **스파이크(spike)**(0/1 이벤트)를 하나씩 띄웁니다. 그래서 정보가 값의 크기뿐 아니라 **언제 발화하는가(타이밍)** 에도 실려, 하나의 신호는 시간에 걸친 이벤트 스트림 형태로 다뤄집니다. 스파이크가 없는 대부분의 순간에는 계산할 것 자체가 없습니다. 이 스파이크 기반 통신을 계산 모델로 옮긴 것이 **스파이킹 신경망(Spiking Neural Network, SNN)**, 이를 전용 칩으로 구현한 것이 Loihi·SpiNNaker 같은 **뉴로모픽(neuromorphic) 하드웨어** 입니다. 드문드문 발화하는 덕에 계산이 희소(sparse)해지고, 그래서 훨씬 적은 전력으로 동작합니다.

이 시리즈는 이 SNN을 **밑바닥부터** 하나씩 직접 만들어 보며 배웁니다. 그 첫걸음으로 이번 편에서는 SNN의 기본 단위인 **스파이킹 뉴런**을 구현하고, 이미지·센서값 같은 실수 데이터를 뉴런이 받을 수 있는 **스파이크로 바꾸는 방법(인코딩)** 을 다룹니다.

## 이번 편에서 배우는 것
1. **LIF 뉴런**을 갱신식 그대로 손으로 구현하고, 막전위·누수·리셋·임계값이 무엇인지 이해한다.
2. 입력 전류와 발화율의 관계(**f–I 곡선**)를 확인한다.
3. 실수 데이터를 스파이크로 바꾸는 세 가지 인코딩: **rate, latency, delta**.
4. 스파이크에서 정보를 되읽는 **디코딩**(스파이크 수, 첫 스파이크 시각).

이제 이 튜토리얼을 실행하기 위해서 내 컴퓨터에 개발 환경을 세팅하고, 필요한 환경 설정과 패키지 설치를 해봅시다.

## 0. 개발 환경

터미널에서 가상환경을 만들고 필요한 패키지를 설치합니다.

```bash
python -m venv .venv
source .venv/bin/activate         # Windows: .venv\Scripts\activate
# 'pip' 명령이 없으면 아래를 python -m pip install ... 로 실행 (pip 자체가 없으면 먼저 python -m ensurepip --upgrade)
pip install torch snntorch numpy matplotlib jupyterlab
```

- `torch`: 텐서 연산 라이브러리.
- `snntorch`: PyTorch 위에서 스파이킹 뉴런 계층(`snn.Leaky` 등)과 인코딩 유틸(`spikegen`)을 제공.
- `numpy`, `matplotlib`: 수치 계산과 그림.

아래 셀은 이 라이브러리들을 불러오고, 재현성(시드)과 그림 설정을 맞추는 **환경 구성 코드**입니다.

In [ ]:
# --- 환경 구성: 라이브러리 불러오기 & 기본 설정 ---
import torch
import snntorch as snn
from snntorch import spikegen
import numpy as np
import matplotlib.pyplot as plt

# 같은 결과가 재현되도록 난수 시드를 고정
torch.manual_seed(42)
np.random.seed(42)

# 한글 폰트 (macOS). 라벨이 네모로 깨지면 'AppleSDGothicNeo' 로 바꿔보세요.
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (9, 3.2)
plt.rcParams["axes.grid"] = True

# 이 노트북 모델은 작아서 CPU로 충분합니다.
device = torch.device("cpu")
print("torch:", torch.__version__, "| snntorch:", snn.__version__, "| device:", device)

## 1. LIF 뉴런: 밑바닥부터

가장 먼저 뉴런 하나를 직접 만들어 봅니다. 생물학적 뉴런은 들어오는 전류를 내부 전압에 모으다가, 그 전압이 일정 수준에 이르면 짧은 전기 펄스(스파이크)를 내보내고 전압이 다시 내려갑니다. **LIF(Leaky Integrate-and-Fire)** 뉴런은 이 동작을 이름 그대로 세 조각으로 모형화합니다.

- **Integrate(적분)**: 입력 전류를 내부 전압에 계속 더해 쌓는다.
- **Leaky(누수)**: 입력이 없으면 쌓인 전압이 시간이 지나며 조금씩 새어 줄어든다.
- **Fire(발화)**: 전압이 임계값에 도달하면 스파이크를 하나 내보내고 전압을 낮춘다(리셋).

### 기호 정리

- **막전위(membrane potential) $V[t]$**: 시각 $t$에 뉴런 내부에 쌓여 있는 전압. 뉴런의 현재 상태를 나타내며, 입력이 들어오면 오르고 가만두면 샙니다. 여기서 $t$는 연속 시간이 아니라 $0, 1, 2, \dots$ 로 세는 **이산 시간 스텝**입니다.
- **입력 전류 $I[t]$**: 시각 $t$에 뉴런으로 들어오는 자극의 세기. 기호 $I$는 물리에서 전류(current)를 가리키는 표준 기호를 그대로 쓴 것입니다. 실제 신경망에서 이 전류는 **앞쪽 뉴런들이 보낸 스파이크에 시냅스 가중치를 곱해 더한 값**입니다.
  $$ I[t] = \sum_i S_i[t]\, w_i \qquad (S_i \in \{0,1\},\; w_i = \text{$i$번 연결의 세기}) $$
  즉 어떤 입력 뉴런 $i$가 그 순간 발화했고($S_i{=}1$) 그 연결이 강하면($w_i$ 큼) 전류가 커집니다. 다만 **이 절에는 아직 앞쪽 뉴런이 없으므로**, $I[t]$를 우리가 직접 정한 값(계단 모양 입력)으로 넣어 뉴런의 반응만 관찰합니다. 스파이크×가중치로 전류를 만드는 실제 회로는 다음 편(02)에서 다룹니다.
- **임계값(threshold) $V_{th}$**: 발화를 일으키는 기준 전압. 학습되는 값이 아니라 우리가 정하는 하이퍼파라미터이며, 관례적으로 $V_{th} = 1.0$으로 두고 입력·가중치 크기를 그에 맞춰 상대적으로 조절합니다.
- **누수 계수 $\beta$ (0~1)**: 매 스텝 막전위에 곱해지는 값. 1에 가까울수록 천천히 새서 과거를 오래 기억하고, 작을수록 빨리 샙니다.
- **스파이크 $S[t]$**: 시각 $t$에 발화했으면 1, 아니면 0.

### 갱신식

이 동작을 이산 시간 갱신식으로 쓰면 LIF의 세 조각과 대응됩니다.

$$ V[t] = \beta\,V[t-1] + I[t] \qquad \text{(Leaky + Integrate: 누수 후 입력 적분)} $$
$$ S[t] = \Theta(V[t]-V_{th}) = \begin{cases} 1 & V[t] \ge V_{th} \\ 0 & \text{그 외} \end{cases} \qquad \text{(Fire)} $$

- 첫 식: 직전 막전위에서 누수되고 남은 $\beta V[t-1]$에 이번 입력 $I[t]$를 더합니다. $\beta<1$이라 과거 값은 매 스텝 조금씩 줄고, 그 위에 새 입력이 쌓입니다.
- 두번째 식은 **발화의 정의**입니다. 막전위가 임계값에 도달하는 순간을 "스파이크가 났다"고 봅니다. 여기 쓰인 계단 모양 함수 $\Theta(x)$를 **헤비사이드 계단함수(Heaviside step function)** 라고 부릅니다(입력이 0 이상이면 1, 아니면 0). 이 표기는 막전위 $V$·임계값 $V_{th}$와 같이 논문에서도 자주 사용하는 형태이며, $V_{th}$를 $\vartheta$로, 막전위 $V$를 $U$로 쓰는 논문도 있습니다.
- **리셋**: 발화한 뒤 막전위를 낮춥니다. 이 과정이 없으면 막전위가 임계값 위에 계속 머물러 매 스텝 발화하게 됩니다. 낮추는 방법은 두 가지가 흔히 쓰입니다.
  - `subtract`: 임계값 $V_{th}$만큼만 빼서 초과분 $V-V_{th}$ 를 남깁니다.
  - `zero`: 0으로 되돌려 초과분을 버립니다.

  리셋에 대해 관례 하나를 정해 둡니다: **리셋은 발화한 바로 다음 스텝에 반영**됩니다. 즉 발화한 스텝의 막전위는 $V_{th}$를 넘긴 값 그대로 남고(이따가 그래프에서 빨간 임계선 위로 뾰족 솟는 모습을 볼 수 있습니다), 그 다음 스텝에서 깎여 내려갑니다. snntorch의 `snn.Leaky`가 바로 이 방식을 쓰기 때문에, 2절에서 `snn.Leaky`로 바꿔도 결과가 스텝 단위로 똑같아집니다. (이 리셋까지 한 줄로 접은 정확한 식은 2절에서 봅니다.)

### 숫자로 손으로 따라가 보기

식이 실제로 어떻게 도는지, $\beta=0.9$, $V_{th}=1.0$, 매 스텝 일정 입력 $I=0.45$로 처음 여섯 스텝을 손으로 계산해 봅니다. 리셋은 **직전 스텝에 발화했으면 이번 스텝 계산에서 $V_{th}$를 빼는** 식으로 반영합니다.

| 스텝 $t$ | 계산: $\beta\,V[t-1] + I - (\text{직전 발화면}\ V_{th})$ | $V[t]$ | 발화? |
|:---:|:---|:---:|:---:|
| 0 | $0.9\cdot 0 + 0.45$ | 0.450 | – |
| 1 | $0.9\cdot 0.450 + 0.45$ | 0.855 | – |
| 2 | $0.9\cdot 0.855 + 0.45$ | 1.220 | ✅ |
| 3 | $0.9\cdot 1.220 + 0.45 - 1.0$ | 0.548 | – |
| 4 | $0.9\cdot 0.548 + 0.45$ | 0.943 | – |
| 5 | $0.9\cdot 0.943 + 0.45$ | 1.299 | ✅ |

두 가지가 눈에 띕니다. 아래 그래프를 읽을 때 이 두 가지를 확인하세요.

- **막전위가 직선이 아니라 곡선으로 오릅니다.** 매 스텝 이전 값에 $\beta(<1)$를 곱하기 때문에, 입력이 일정해도 증가폭이 점점 줄어(+0.450 → +0.405 → +0.365 …) 어떤 값(여기선 $I/(1-\beta)=4.5$)을 향해 **완만히 휘며** 다가갑니다. 이 누수가 만드는 곡률이 곧 아래 그래프에서 보게 될 굴곡입니다.
- **2스텝에서 $V_{th}=1.0$을 넘겨 발화**하면, 그 값(1.220)은 그대로 두고 **다음 스텝(3)에서 $V_{th}$를 빼** 0.548로 내려갑니다. 쌓임 → 발화 → (다음 스텝)리셋 → 다시 쌓임의 리듬이 반복됩니다.

In [ ]:
# --- 하이퍼파라미터 ---
beta, threshold = 0.9, 1.0   # 누수 계수 β, 임계값 V_th
T = 150                      # 시뮬레이션할 총 시간 스텝 수 (t = 0, 1, ..., T-1)

# 입력 전류 I[t]: 시간에 따라 두 단계로 세지는 계단 입력.
# (실제로는 앞쪽 뉴런의 스파이크×가중치이지만, 이 절에선 값을 직접 지정한다.)
cur = torch.zeros(T)         # 길이 T 배열, 각 칸이 한 시간 스텝의 입력 I[t]
cur[20:70]  = 0.20           # 20~69스텝: 약한 입력
cur[80:140] = 0.45           # 80~139스텝: 강한 입력

def run_lif(reset):
    # 위 규칙을 그대로 구현. 리셋은 '직전 스텝에 발화했으면 이번 스텝에 반영'(snn.Leaky와 동일).
    V = 0.0                                        # V[-1] = 0 에서 시작
    S_prev = 0.0                                   # 직전 스텝 스파이크 (리셋에 사용)
    V_rec, S_rec = [], []
    for t in range(T):
        # (1) Leaky + Integrate, 그리고 직전 발화분 리셋을 함께 반영
        if reset == "subtract":
            V = beta * V + cur[t].item() - S_prev * threshold   # 직전에 발화했으면 V_th만큼 뺀다
        else:  # "zero": 직전에 발화했으면 먼저 0으로 지운 뒤 이번 입력을 더한다
            V = beta * (1.0 - S_prev) * V + cur[t].item()
        # (2) Fire: 헤비사이드 계단함수 Θ(V - V_th)
        S = 1.0 if V >= threshold else 0.0
        V_rec.append(V); S_rec.append(S)
        S_prev = S
    return np.array(V_rec), np.array(S_rec)

mem_sub,  spk_sub  = run_lif("subtract")
mem_zero, spk_zero = run_lif("zero")
print("subtract 스파이크:", int(spk_sub.sum()), "| zero 스파이크:", int(spk_zero.sum()))

# 강한 입력 구간(80~139)의 발화 간격(ISI, inter-spike interval)을 수치로 비교
def strong_isi(spk):
    ts = np.where(spk > 0)[0]; ts = ts[(ts >= 80) & (ts < 140)]
    return np.diff(ts)                            # 이웃한 두 발화 사이의 스텝 수
isi_sub, isi_zero = strong_isi(spk_sub), strong_isi(spk_zero)
print(f"강한 구간 평균 발화간격(ISI) — subtract: {isi_sub.mean():.2f}스텝, zero: {isi_zero.mean():.2f}스텝")

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(9, 6.6),
                       gridspec_kw={"height_ratios": [0.8, 1.7, 1.2, 0.9]})
n = len(cur)   # 축 범위는 데이터 길이로 (뒤 절에서 T가 바뀌어도 이 그림은 영향 없음)

# (0) 입력 전류
ax[0].plot(cur.numpy(), color="tab:gray"); ax[0].set_ylabel("입력 I[t]")
ax[0].set_title("리셋 방식 비교: 같은 입력, 다른 막전위 궤적")

# (1) 막전위 전체 — 값은 정수 스텝에서만 정의되는 이산 표본이라 점으로, 선은 궤적 보조선
w0, w1 = 80, 100
ax[1].axvspan(w0, w1 - 1, color="0.85", alpha=0.5, zorder=0)          # 아래에서 확대할 구간 표시
ax[1].plot(mem_sub,  color="tab:blue",   marker=".", ms=2.5, lw=0.9, label="subtract")
ax[1].plot(mem_zero, color="tab:orange", marker=".", ms=2.5, lw=0.9, alpha=0.85, label="zero")
ax[1].axhline(threshold, ls="--", c="r")
ax[1].text(n - 2, threshold + 0.06, "임계값 $V_{th}$", color="r", fontsize=8, ha="right")
ax[1].set_ylabel("막전위 V[t]"); ax[1].legend(loc="upper left", ncol=2, fontsize=8)

# (2) 위 회색 구간을 확대: 점 하나가 한 스텝 (발화 뒤 다음 스텝에 리셋으로 떨어지는 게 보인다)
ax[2].plot(range(w0, w1), mem_sub[w0:w1],  color="tab:blue",   marker="o", ms=4, lw=0.9)
ax[2].plot(range(w0, w1), mem_zero[w0:w1], color="tab:orange", marker="o", ms=4, lw=0.9, alpha=0.85)
ax[2].axhline(threshold, ls="--", c="r"); ax[2].axhline(0, ls=":", c="gray", lw=0.8)
ax[2].set_ylabel("V[t] 확대"); ax[2].set_xlim(w0 - 0.5, w1 - 0.5)
ax[2].set_title(f"↑ 회색 구간 확대: {w0}~{w1 - 1}스텝 (점 하나 = 한 스텝)", fontsize=9)

# (3) 스파이크
ax[3].eventplot(np.where(spk_sub > 0)[0],  colors="tab:blue",   lineoffsets=1, linelengths=0.7)
ax[3].eventplot(np.where(spk_zero > 0)[0], colors="tab:orange", lineoffsets=0, linelengths=0.7)
ax[3].set_yticks([0, 1]); ax[3].set_yticklabels(["zero", "subtract"])
ax[3].set_ylabel("스파이크"); ax[3].set_xlabel("시간 스텝 t")

for a in (ax[0], ax[1], ax[3]):
    a.set_xlim(0, n)
ax[0].tick_params(labelbottom=False); ax[1].tick_params(labelbottom=False)
plt.tight_layout(); plt.show()

### 🔎 그림 읽기

- **왜 막전위가 곧게 오르지 않고 완만히 휘나요?** 누수 때문입니다. 매 스텝 이전 막전위에 $\beta=0.9$를 곱하므로, 입력이 일정해도 오르는 폭이 점점 줄어 $I/(1-\beta)$라는 천장을 향해 곡선으로 다가갑니다(위 표 그대로). $\beta$를 1로 두면(누수 없음) 직선에 가까워지고, 작게 두면 더 빨리 휩니다.
- **점인가요, 선인가요?** 막전위는 정수 스텝 $t=0,1,2,\dots$에서만 정의되는 **이산 신호**라 정확히는 점(표본)입니다. 확대 그림의 동그라미 하나가 한 스텝이고, 이어진 선은 궤적을 눈으로 따라가기 위한 보조선일 뿐 스텝과 스텝 사이에 실제 값이 있는 것은 아닙니다.
- **빨간 임계선 위로 솟은 점은 뭔가요?** 발화한 스텝의 막전위입니다. 리셋이 다음 스텝에 반영되므로, 발화한 그 스텝에서는 $V_{th}$를 넘긴 값이 그대로 표본에 남아 빨간 선 위로 뾰족 올라옵니다. 이 뾰족한 점의 시각이 맨 아래 스파이크 그림의 점과 정확히 일치합니다.
- **`zero`는 발화 후 왜 바닥(0)에 안 닿아 보이나요?** zero 리셋은 발화한 **다음 스텝**에 막전위를 0으로 지운 뒤, 곧바로 그 스텝의 입력 $I[t]$를 더합니다. 그래서 표본으로 찍히는 값은 0이 아니라 $I[t]$(강한 입력 구간에선 0.45)에서 다시 출발합니다. 0은 입력이 더해지기 직전의 '순간'이라 스텝 표본에는 잡히지 않습니다. 확대 그림의 회색 점선(0) 위에 주황 점들이 앉지 않고 살짝 떠 있는 이유가 이것입니다.

### 🔎 조금 더 들어가기: 리셋 방식이 만드는 차이

두 리셋의 차이는 **발화 순간의 '초과분'을 어떻게 처리하느냐** 하나뿐입니다. 위 손계산표의 2스텝에서 막전위는 $V=1.220$ 으로 임계값 $V_{th}=1.0$ 을 $0.220$ 만큼 넘겼습니다(이 $0.220$ 이 초과분).

- **subtract**: 초과분을 버리지 않고 $V_{th}$ 만큼만 빼서 **다음 스텝으로 넘깁니다.**
- **zero**: 막전위를 통째로 0으로 지워 **초과분을 버리고**, 그 스텝의 입력부터 다시 쌓기 시작합니다(직전 상태를 완전히 잊음).

같은 입력($I=0.45$)에 두 방식을 나란히 굴리면, 첫 발화 뒤부터 궤적이 갈립니다.

| 스텝 $t$ | subtract $V$ (발화?) | zero $V$ (발화?) |
|:--:|:--:|:--:|
| 2 | 1.220 ✅ | 1.220 ✅ |
| 3 | 0.548 | 0.450 |
| 4 | 0.943 | 0.855 |
| 5 | 1.299 ✅ | 1.220 ✅ |
| 6 | 0.619 | 0.450 |
| 7 | **1.007 ✅** | 0.855 |
| 8 | 0.356 | **1.220 ✅** |

2스텝에서 함께 발화한 뒤, subtract는 넘겨받은 초과분 덕에 매 주기 **조금 높은 곳에서 출발**합니다(3스텝: 0.548 vs 0.450). 이 여유가 쌓여 **7스텝에서 subtract가 한 발 먼저** 터지고, zero는 8스텝에야 터집니다. 그래서 강한 입력 구간에서 subtract는 이따금 **2스텝 간격**으로도 발화(평균 발화간격 ISI ≈ 2.76스텝)하는 반면, zero는 초과분을 매번 버려 **3스텝 간격**으로 규칙적입니다(≈ 3.00스텝). 위 코드가 출력한 스파이크 수 28 대 27, 이 **한 발 차이**가 바로 여기서 옵니다(눈으로는 잘 안 보여 ISI 수치로 확인).

한 가지만 덧붙입니다. "어느 쪽이 더 촘촘한가"는 보편 법칙이 아니라 **입력 세기와 이산 스텝이 맞물리는 방식에 따라 달라지고, 더 약하거나 훨씬 강한 입력에서는 순위가 뒤집히기도** 합니다. 여기서 기억할 핵심은 순위가 아니라, **리셋 규칙 하나(초과분을 넘기느냐 버리느냐)가 발화 시점을 바꾼다**는 점입니다. SNN에서 "언제 발화하는가"는 이런 작은 규칙들이 결정합니다.

## 2. f–I 곡선: 입력 전류와 발화율

1절에서는 뉴런 하나에 계단 입력을 주고 막전위가 쌓여 발화하는 과정을 눈으로 봤습니다. 이번에는 질문을 하나로 좁힙니다: **입력 전류를 세게 줄수록 뉴런은 실제로 얼마나 더 자주 발화할까?** 일정한 입력 전류 $I$를 여러 값으로 바꿔가며 각 경우의 발화율을 재서 그린 그래프가 **f–I 곡선**입니다. 여기서 **f**는 발화율(firing rate, 단위 시간당 스파이크 수), **I**는 입력 전류(current)로, 그래프의 세로축과 가로축 이름이 곧 곡선의 이름이 됩니다.

이걸 재려면 같은 뉴런을 입력마다 여러 번 돌려야 하는데, 1절처럼 규칙을 매번 손으로 쓰는 대신 snntorch가 제공하는 기성 LIF 계층 **`snn.Leaky`** 를 씁니다. `snn.Leaky`는 1절에서 우리가 손으로 짠 뉴런과 **똑같은** 것을 `beta`, `threshold`, `reset_mechanism` 인자로 만들어 줍니다(아래 코드의 `lif` 변수가 그 뉴런입니다).

### snn.Leaky가 실제로 쓰는 한 줄짜리 식

1절에서 우리는 '적분+누수 → 발화 → 리셋'을 나눠 설명했습니다. `snn.Leaky`는(그리고 많은 논문·자료가) 이 리셋까지 포함해 **막전위 갱신을 한 줄**로 접어 씁니다(subtract 리셋 기준 — `snn.Leaky` 기본값이자 아래 f–I에서 쓰는 방식):

$$ V[t] = \beta\,V[t-1] + I[t] - S[t-1]\,V_{th} $$

- 앞의 $\beta V[t-1] + I[t]$ 는 1절의 **누수 + 적분** 그대로입니다.
- 마지막 항 $-\,S[t-1]\,V_{th}$ 이 **리셋**입니다 — *"직전 스텝에 발화했다면($S[t-1]=1$) 이번 스텝에서 $V_{th}$만큼 뺀다"*.
- 발화 판정 $S[t] = \Theta(V[t]-V_{th})$ 는 이 줄에 안 보이지만 매 스텝 함께 계산됩니다.

리셋이 $S[t]$가 아니라 **$S[t-1]$**(직전 스파이크)로 붙는 건 발화가 한 스텝 늦게 반영되기 때문인데, 이게 1절에서 정해 둔 "리셋은 다음 스텝에" 관례이자 `snn.Leaky`의 기본 동작입니다. 그래서 1절 손구현과 `snn.Leaky`의 막전위·스파이크가 스텝 단위로 일치합니다.

자료에 따라 같은 수식에 다른 기호를 쓰기도 합니다. 막전위를 $u$·$U$로, 누수 계수를 $\lambda$로, 스파이크를 $s$로, 리셋으로 빼는 값을 $V_{\text{reset}}$로 두면 $u_t = \lambda\,u_{t-1} + I_t - s_{t-1}\,V_{\text{reset}}$ 처럼 보이지만(여기 subtract에선 $V_{\text{reset}}=V_{th}$), 전부 위와 같은 식입니다. 다른 기호 예시들은 다음과 같습니다.

| 이 문서 | 논문에서 흔한 다른 기호 | 뜻 |
|---|---|---|
| $V[t]$ | $u_t,\ U_t$ | 막전위 |
| $\beta$ | $\lambda$ | 누수 계수 |
| $S[t]$ | $s_t,\ z_t$ | 스파이크 (0/1) |
| $V_{th}$ | $\vartheta,\ V_{thr}$ | 임계값 |
| $\Theta$ | $H$ | 헤비사이드 계단함수 |

### 한 스텝 실행하는 법

이 뉴런을 **한 스텝** 굴리는 호출은 다음 꼴입니다.

```python
spk, mem = lif(I, mem)   # (입력 전류, 직전 막전위) -> (이번 스파이크, 갱신된 막전위)
```

**입력 전류 `I`와 직전 막전위 `mem`을 넣으면, 그 스텝의 스파이크 `spk`와 갱신된 막전위 `mem`을 돌려줍니다.** 위 한 줄짜리 식 한 번 + 발화 판정 한 번을 이 호출이 대신하는 셈이고, 반환된 `mem`을 다음 호출에 도로 넣어 주면 1절의 루프와 똑같이 시간을 이어 갑니다.

이렇게 얻는 f–I 곡선은 일반 신경망의 ReLU/GELU 같은 활성화 함수에 해당합니다. 입력을 출력으로 바꾼다는 역할은 같고, 그 출력이 하나의 실수가 아니라 시간에 걸친 발화율로 나타난다는 점이 다릅니다.

- 입력이 임계값을 넘길 만큼 크지 않으면 막전위가 $V_{th}$에 못 미쳐 발화율은 0입니다.
- 임계값을 넘으면 입력이 셀수록 더 자주 발화해 발화율이 대체로 단조 증가합니다.

이 관계가 SNN이 신호의 세기를 표현하는 기본 방식(rate coding, 3절)의 바탕입니다.

In [ ]:
beta, threshold = 0.9, 1.0
T = 200                                    # 각 입력 전류마다 이만큼의 스텝을 돌려 발화율을 측정
currents = torch.linspace(0.0, 0.6, 40)    # 0.0~0.6 사이 40개의 '일정 입력 전류' 후보

rates = []
for I in currents:
    # 입력 전류 하나당 새 뉴런을 만들어 막전위 0에서 다시 시작 (이전 실험의 잔여 상태 제거)
    lif = snn.Leaky(beta=beta, threshold=threshold, reset_mechanism="subtract")
    mem = torch.zeros(1)                   # 막전위 초기값 V[-1] = 0
    n = 0                                  # 이 입력에서 T스텝 동안 센 총 스파이크 수
    for t in range(T):
        # (입력 전류, 직전 막전위) -> (이번 스파이크, 갱신된 막전위).  §1 손구현과 문자 그대로 같은 계산.
        spk, mem = lif(I.reshape(1), mem)
        n += int(spk.item())
    rates.append(n / T)                    # 발화율 f = 스파이크 수 / 스텝 수 (스텝당 평균 스파이크)

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(currents.numpy(), rates, marker="o", ms=3, color="tab:green")
ax.set_xlabel("일정 입력 전류 I"); ax.set_ylabel("발화율 f (스파이크/스텝)")
ax.set_title(f"LIF의 f–I 곡선 (β={beta}, Vth={threshold})")
plt.tight_layout(); plt.show()
print(f"발화 시작 지점(임계 전류) 근처: I ≈ {currents[np.argmax(np.array(rates)>0)]:.3f}")

## 3. 스파이크 인코딩: 실수 데이터 → 스파이크

지금까지 만든 뉴런은 입력으로 **전류(또는 스파이크)** 만 받습니다. 그런데 우리가 다루고 싶은 데이터—이미지 픽셀, 센서 측정값—는 대부분 그냥 실수입니다. 그래서 SNN에 무언가를 넣으려면 먼저 그 실수 값을 **스파이크 열(0/1이 시간축을 따라 늘어선 것)로 바꿔야** 합니다. 이 변환이 **인코딩(encoding)** 입니다.

전체 그림은 이렇습니다: **실수 데이터 →(인코딩)→ 스파이크 →(뉴런이 처리)→ 스파이크 →(디코딩)→ 값.** 이 절은 첫 화살표(인코딩)를, 4절은 마지막 화살표(디코딩, 스파이크에서 값을 되읽기)를 다룹니다. 그래서 지금 스파이크를 만드는 법을 익혀 두면 4절에서 그것을 되읽는 과정이 자연스럽게 짝을 이룹니다.

값을 스파이크의 어느 성질에 싣느냐에 따라 세 가지 방식이 있습니다.

| 방식 | 정보를 싣는 곳 | 직관 | snntorch |
|---|---|---|---|
| **Rate** | 스파이크 **개수/빈도** | 값이 클수록 자주 발화 | `spikegen.rate` |
| **Latency** | 첫 스파이크 **타이밍** | 값이 클수록 빨리 발화 | `spikegen.latency` |
| **Delta** | 값의 **변화** | 신호가 확 바뀔 때만 발화 (이벤트 카메라) | `spikegen.delta` |

입력 예제로 쓸 **합성 이미지** 하나를 만듭니다. 12×12 크기에 중앙이 밝은 가우시안 blob이고 픽셀 값은 0~1입니다. 이 이미지를 SNN에 넣기 위한 밑작업으로, **144개(=12×12) 픽셀을 한 줄로 펴서(flatten) 길이 144 벡터**로 만듭니다. 픽셀 하나가 입력 뉴런 하나에 대응한다고 보면 됩니다. 다음 절들에서 이 144개 값을 각각 스파이크 열로 바꿔, 최종적으로 `[T, 144]`(시간 × 픽셀) 모양의 스파이크 텐서를 얻습니다 — 이 텐서가 바로 SNN에 흘려 넣는 입력입니다.

In [ ]:
# 12x12 합성 이미지: 중앙이 밝은 가우시안 blob (값 0~1)
g = 12
yy, xx = np.meshgrid(np.linspace(-1, 1, g), np.linspace(-1, 1, g), indexing="ij")
img = np.exp(-((xx - 0.15)**2 + (yy + 0.1)**2) / 0.35).astype(np.float32)
img = (img - img.min()) / (img.max() - img.min())   # 0~1로 정규화
img_t = torch.tensor(img).flatten()   # 12x12 이미지를 길이 144 벡터로 펴기 (픽셀 1개 = 입력 뉴런 1개)

fig, ax = plt.subplots(figsize=(3.2, 3.2))
im = ax.imshow(img, cmap="magma"); ax.set_title("원본 이미지 (12x12, 값 0~1)")
ax.set_xticks([]); ax.set_yticks([]); plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

### 3-1. Rate coding — 값을 발화 확률로

이제 입력 예제를 만들었으니, 입력을 인코딩 해 봅시다. Rate coding은 인코딩 중 가장 단순한 방식입니다. 각 픽셀 값을 **발화 확률**로 해석해서, 매 시간 스텝마다 그 확률로 동전을 던지듯 스파이크를 낼지(1) 말지(0) 정합니다(베르누이 시행). 예를 들어 값이 0.8인 픽셀은 매 스텝 약 80% 확률로, 0.1인 픽셀은 약 10% 확률로 스파이크를 냅니다. 그래서 **밝은(값이 큰) 픽셀일수록 시간축을 따라 스파이크가 촘촘**해집니다.

아래 코드의 `spikegen.rate(img_t, num_steps=T)`가 바로 이 일을 합니다. 길이 144 벡터를 받아 `T`스텝 동안 픽셀마다 베르누이 시행을 반복하여 `[T, 144]` 스파이크 텐서를 돌려줍니다. 거꾸로 이 스파이크를 시간축으로 평균 내면 각 픽셀의 발화 빈도가 원래 값에 가까워지므로, 원본이 (잡음이 섞인 채) 복원됩니다.

In [ ]:
T = 100
rate_spk = spikegen.rate(img_t, num_steps=T)        # [T,144]: 픽셀마다 매 스텝 확률 p=픽셀값 으로 베르누이 발화
recon = rate_spk.float().mean(0).reshape(g, g)      # 시간축 평균 → 각 픽셀의 발화 빈도 ≈ 원래 값 (복원)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
ax[0].imshow(img, cmap="magma"); ax[0].set_title("원본"); ax[0].axis("off")
ax[1].imshow(recon.numpy(), cmap="magma"); ax[1].set_title(f"스파이크 시간평균 복원 (T={T})"); ax[1].axis("off")

# 대표 픽셀 24개의 래스터: 밝은 픽셀(위)일수록 스파이크가 촘촘한지 확인
idx = torch.argsort(img_t, descending=True)[torch.linspace(0, 143, 24).long()]
rows = [np.where(rate_spk[:, i].numpy() > 0)[0] for i in idx]
ax[2].eventplot(rows, colors="k", linelengths=0.8)
ax[2].set_title("rate 래스터 (위=밝은 픽셀)"); ax[2].set_xlabel("시간 스텝"); ax[2].set_ylabel("픽셀(밝기순)")
ax[2].invert_yaxis()   # 행 0(가장 밝은 픽셀)을 맨 위로 (latency 래스터와 방향 통일)
plt.tight_layout(); plt.show()

### 3-2. Latency coding — 값을 첫 스파이크 시각으로

rate가 "얼마나 자주"에 정보를 실었다면, latency는 "언제"에 실어서 입력을 인코딩합니다. 각 픽셀은 관측 구간 동안 **딱 한 번만** 발화하는데, 그 **한 번의 시각**이 값에 따라 정해집니다: 값이 클수록 일찍(작은 $t$에), 작을수록 늦게 발화합니다. 값이 0에 가까운 픽셀은 아주 늦게 발화하거나 관측 구간 안에 아예 나오지 않기도 합니다.

`spikegen.latency`는 값→시각 변환에 지수 함수를 쓰며, 그 곡률을 시간상수 `tau`로 조절합니다. 결과만 보면 **밝은 픽셀이 먼저, 어두운 픽셀이 나중에** 한 발씩 켜집니다. 정보가 스파이크 '개수'가 아니라 '타이밍'에 실리므로 rate보다 스파이크가 훨씬 적습니다(픽셀당 최대 1개). 이 희소함이 latency의 장점입니다: 같은 정보를 훨씬 적은 스파이크로 전달하므로 계산·에너지가 절약됩니다. 픽셀별 첫 스파이크 시각을 이미지로 그리면 밝은 곳이 먼저 켜지는 것이 한눈에 보입니다.

In [ ]:
T = 60
# spikegen.latency: 픽셀당 최대 1개 스파이크, 값이 클수록 이른 시각에 발화.
#   tau=값→시각 변환의 시간상수, threshold=이보다 작은 값은 무시(발화 안 함),
#   normalize=발화 시각을 [0,T) 구간에 맞춤, linear=시각 간격을 선형화, clip=범위 밖 제거
# [T,144]로, 60 시간스텝과 144픽셀수로 구성됨. lat_spk[t, i] = "시각 t에 픽셀 i가 발화했으면 1, 아니면 0"
lat_spk = spikegen.latency(img_t, num_steps=T, tau=5, threshold=0.01,
                           clip=True, normalize=True, linear=True)

# first는 픽셀별 첫 스파이크 시각을 담는 배열, 처음에는 전부 T(=60)으로 채워놓고, 스파이크가 있으면 그 시각으로 낮춰 갱신
# 한 번도 발화 안 하면 여전히 T로 둠. 그래서 first=60은 관측구간 [0,60)에서 한 번도 안 터졌다는 뜻.
first = np.full(img_t.numel(), T, dtype=float)
ts, us = torch.where(lat_spk > 0)
for t, u in zip(ts.tolist(), us.tolist()):
    if t < first[u]:
        first[u] = t
first_img = first.reshape(g, g)

fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
ax[0].imshow(img, cmap="magma"); ax[0].set_title("원본"); ax[0].axis("off")
im = ax[1].imshow(first_img, cmap="viridis_r")
ax[1].set_title("첫 스파이크 시각 (밝을수록 빨리)"); ax[1].axis("off"); plt.colorbar(im, ax=ax[1], fraction=0.046)
idx = torch.argsort(img_t, descending=True)[torch.linspace(0, 143, 24).long()]
rows = [np.where(lat_spk[:, i].numpy() > 0)[0] for i in idx]
ax[2].eventplot(rows, colors="tab:red", linelengths=0.8)
ax[2].set_title("latency 래스터 (위=밝은 픽셀)"); ax[2].set_xlabel("시간 스텝"); ax[2].set_ylabel("픽셀(밝기순)")
ax[2].invert_yaxis()   # 행 0(가장 밝은 픽셀)을 맨 위로 → 제목의 "위=밝은 픽셀"과 일치
plt.tight_layout(); plt.show()
n_rate = int(spikegen.rate(img_t, num_steps=T).sum())   # 같은 이미지를 rate 코딩했을 때 총 스파이크 수
n_lat  = int(lat_spk.sum())                             # latency 코딩의 총 스파이크 수
print(f"같은 {img_t.numel()}개 픽셀을 {T}스텝 동안 인코딩 — rate: {n_rate}개, latency: {n_lat}개")

### 🔎 rate vs latency
이렇게 이미지, 그러니까 실수 데이터를 스파이크로 바꾸는 인코딩을 해보았습니다. 아직 학습을 시킨 건 아니고, 형식 변환을 한 것입니다.
spike들의 반응을 통해 나온 이미지를 다시 원본과 대조한 것은, 인코딩이 정보를 안 잃었다는 사실을 눈으로 확인한 것일 뿐, 아직 최적화는 하지 않았습니다. 우리가 이미지 인코딩을 예시로 든 이유는 실수 데이터를 스파이크로 확인하기에 직관적이기 때문입니다.

이제 rate와 latency 인코딩의 차이를 한번 분석해 봅시다. 같은 144개 픽셀·같은 스텝을 인코딩해도 **rate는 픽셀당 매 스텝 확률적으로 여러 번**, **latency는 픽셀당 최대 1번** 발화합니다(위 출력 참고). 그래서 이 예시의 경우 latency의 총 스파이크가 약 15배 적습니다 — **정보를 개수가 아니라 타이밍에 싣기 때문**입니다.

이 latency가 rate에 대해 가지는 희소함(sparsity)은 장단점이 있습니다.

**latency의 장점**
- **에너지·계산 절약**: 스파이크 하나가 곧 연산·전력이라, 적을수록 (특히 뉴로모픽 칩에서) 효율적입니다.
- **빠른 판단**: 첫 스파이크만 보고 결정할 수 있어, 전체 시간창을 다 기다릴 필요가 없습니다.

**latency의 단점**
- **잡음에 취약**: 정보가 정밀한 발화 *시각*에 실려서, 시각이 조금만 흔들려도 값이 틀어집니다.
- **중복이 없음**: 픽셀당 스파이크가 하나뿐이라 그 하나가 어긋나면 정보를 잃습니다. (rate는 여러 스파이크를 평균 내므로 잡음에 강한 대신 스파이크가 많습니다.)

즉 **에너지·속도가 중요하면 latency, 잡음에 견고해야 하면 rate** — 과제와 하드웨어에 따라 고르면 되는 것입니다.

### 3-3. Delta coding — 값의 변화에만 발화

rate와 latency는 **정지된 값 하나**(정지 이미지의 픽셀)를 인코딩했습니다. delta는 다릅니다 — **시간에 따라 변하는 신호**를 받아 "값이 얼마나 변했는가"에 발화합니다. 직전 값보다 임계 이상 **커지면 양(+) 스파이크**, 임계 이상 **작아지면 음(−) 스파이크**, 변화가 작으면 **침묵**합니다.

그래서 입력도 이미지가 아니라 **시간에 따라 오르내리는 1차원 신호** `[T, 1]`이 필요합니다. 예시로는 **진폭이 점점 커지는 사인파**를 줍니다 — 시간에 따라 값이 변하는 센서 측정값이라고 보면 됩니다. 진폭이 커질수록 값이 더 가파르게 변하고, 그때 delta가 **더 자주** 발화하는 것을 보게 됩니다. 그래서 delta로 인코딩된 데이터는 매우 **희소**하고, 변화가 생기는 즉시 나오므로 **빠릅니다**.

In [ ]:
T = 120
t_axis = torch.linspace(0, 4*np.pi, T)
signal = (torch.sin(t_axis) * (t_axis / (4*np.pi))).unsqueeze(1)   # 예시 입력: 진폭이 점점 커지는 사인파 (시간에 따라 오르내리는 1D 신호 = 센서값의 대역) [T,1]
# threshold=변화가 이보다 커야 발화, off_spike=True 면 감소 방향에도 음(-) 스파이크를 냄
delta_spk = spikegen.delta(signal, threshold=0.05, off_spike=True)  # +1 / -1 스파이크 (신호의 스텝 변화가 이 값을 넘을 때만)

fig, ax = plt.subplots(2, 1, figsize=(9, 3.8), sharex=True)
ax[0].plot(signal[:, 0].numpy(), color="tab:gray"); ax[0].set_ylabel("입력 신호")
ax[0].set_title("Delta coding: 변화가 클 때만 스파이크")
up = np.where(delta_spk[:, 0].numpy() > 0)[0]; dn = np.where(delta_spk[:, 0].numpy() < 0)[0]
ax[1].eventplot(up, colors="tab:blue", lineoffsets=1, linelengths=0.7, label="ON (증가)")
ax[1].eventplot(dn, colors="tab:red", lineoffsets=0, linelengths=0.7, label="OFF (감소)")
ax[1].set_yticks([0, 1]); ax[1].set_yticklabels(["OFF", "ON"]); ax[1].set_xlabel("시간 스텝"); ax[1].legend(loc="upper left")
plt.tight_layout(); plt.show()

## 4. 디코딩: 스파이크에서 정보 되읽기

3절의 인코딩이 **값 → 스파이크** 였다면, 디코딩은 그 반대인 **스파이크 → 값**입니다. SNN의 마지막 층이 내놓는 것도 결국 스파이크 열이므로, "그래서 답이 뭔데?"를 알려면 이 스파이크를 다시 사람이 읽을 수 있는 값(예: 클래스 번호)으로 되돌려야 합니다. 뒤 편들에서 뉴런의 출력을 판단하거나 학습 신호로 쓸 때 이 과정이 필요합니다.

3절의 인코딩 방식과 짝을 이루는 두 가지 대표 방법이 있습니다.

- **스파이크 수(rate readout)**: 시간창 동안 각 출력 뉴런의 스파이크를 세고, 가장 많이 발화한 뉴런을 답으로 봅니다. rate coding과 짝을 이룹니다.
- **첫 스파이크 시각(latency readout)**: 어느 뉴런이 **가장 먼저** 발화했는지를 봅니다. latency coding과 짝을 이루며, 전체 시간창을 다 기다릴 필요 없이 몇 스텝 만에 판단할 수 있어 추론이 빠릅니다.

아래 예제는 분류기의 출력층을 흉내 냅니다. 세 개의 클래스 뉴런에 서로 다른 발화 확률(0.1 / 0.25 / 0.6)을 주어 rate coding으로 스파이크 열을 만든 뒤, 두 디코딩 방식이 모두 **가장 활발한 뉴런(class 2)** 을 정답으로 가리키는지 확인합니다. 스파이크 수 방식은 개수의 최댓값(`argmax`)을, 첫 스파이크 시각 방식은 시각의 최솟값(`argmin`, 가장 먼저 발화한 뉴런)을 답으로 고릅니다.

In [ ]:
T = 80
true_class = 2
rates3 = torch.tensor([0.1, 0.25, 0.6])          # 세 클래스의 발화 확률 (class 2가 가장 활발)
out_spk = spikegen.rate(rates3, num_steps=T)     # [T,3]: 각 클래스를 rate coding으로 발화

count = out_spk.sum(0)                            # 디코딩 ① rate readout: 클래스별 총 스파이크 수
# 디코딩 ② latency readout: 클래스별 '첫' 스파이크 시각 (한 번도 발화 안 하면 T로 둠)
first_t = torch.tensor([ (torch.where(out_spk[:, c] > 0)[0][:1].item()
                          if out_spk[:, c].sum() > 0 else T) for c in range(3)])
print("스파이크 수 :", count.tolist(),   "-> 예측(argmax, 최다 발화):", int(count.argmax()))
print("첫 스파이크 :", first_t.tolist(), "-> 예측(argmin, 최초 발화):", int(first_t.argmin()))
print("정답 클래스 :", true_class)

fig, ax = plt.subplots(figsize=(8, 2.6))
rows = [np.where(out_spk[:, c].numpy() > 0)[0] for c in range(3)]
ax.eventplot(rows, colors=["tab:gray", "tab:gray", "tab:green"], linelengths=0.7)
ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["class0", "class1", "class2(정답)"])
ax.set_xlabel("시간 스텝"); ax.set_title("3-클래스 출력: 스파이크 수 / 첫 스파이크로 디코딩")
plt.tight_layout(); plt.show()

## 5. 정리 & 다음 단계

### 배운 것
- **LIF 뉴런**: 입력을 막전위에 쌓고(Integrate), 시간에 따라 새고(Leaky), 임계값에서 발화 후 리셋(Fire)한다. 손으로 구현했고, 이것이 `snn.Leaky`가 쓰는 한 줄짜리 식 $V[t] = \beta\,V[t-1] + I[t] - S[t-1]\,V_{th}$ 과 같음을 확인했다.
- 리셋 방식(subtract, zero)이 발화 시점을 바꾼다.
- **f–I 곡선**: 뉴런은 입력 전류를 발화율로 바꾸는 비선형이다.
- **인코딩 3종**: rate(개수), latency(타이밍, 희소), delta(변화, 초희소). **디코딩**: 스파이크 수, 첫 스파이크 시각.

### 한 가지 포인트: 계단 함수와 학습
발화를 정하는 헤비사이드 계단함수 $\Theta$는 미분이 (임계점을 빼면) 어디서나 0이라, 일반 신경망처럼 기울기를 역전파해 학습하기가 까다롭습니다. 이를 우회하는 대표적 방법이 역전파 때만 미분을 매끄러운 함수로 대신하는 **대체 기울기(surrogate gradient)** 이지만, 이 시리즈에서는 먼저 실제 뇌가 작동하는 원리와 더 비슷한, 시냅스가 국소 신호로 스스로 배우는 **생물학적 학습 규칙**(다음 편 STDP부터)을 배워볼 것 입니다. 지금은 "스파이크 함수는 미분이 안 되어 학습이 특별하다"는 사실만 씨앗으로 담아 두면 됩니다. 미분 우회를 사용하는 방법들은 이후 Ch04에서 다룰 것입니다.

### 직접 바꿔보면 좋은 것
- 1절 `run_lif`의 `beta`를 `0.5`로 낮춰 보기 → 막전위가 빨리 새서 발화가 줄어드는지.
- latency 셀의 `tau`를 `1`, `10`으로 → 첫 스파이크 시각 분포가 어떻게 퍼지는지.
- delta 셀의 `threshold`를 `0.03`(더 촘촘)과 `0.10`(더 희소)으로 → 스파이크가 얼마나 촘촘/희소해지는지. (이 신호는 스텝 변화가 최대 ~0.10이라 그보다 크면 거의 침묵)

### 이제 Ch02로 넘어갈 준비가 되었습니다
이제 데이터를 스파이크로 바꿀 수 있으니, 다음 편에서는 정답 라벨 없이 시냅스가 스스로 패턴을 학습하는 첫 규칙 **STDP**로 갑니다. "먼저 발화한 입력이 출력의 발화를 도왔다면 그 연결을 강화한다"는 규칙을, 앞에서처럼 직접 구현해 확인합니다.

*생물학적으로 더 타당한(biologically plausible) 뉴로모픽 학습에 왜 관심을 가져야 하는지 궁금하다면, 바로 이어지는 6절을 읽어 보세요.*

## 6. 부록 · 한 걸음 물러서서 — 왜 '생물학적 학습'인가

바로 위에서 계단 함수에 대해 언급을 했습니다. 스파이크를 정하는 계단함수는 미분이 잘 안 되어서, 일반 신경망처럼 전역 오차를 역전파하는 대신 이 시리즈는 **시냅스가 자기 주변의 국소 신호만으로 스스로 배우는 생물학적 학습 규칙**을 주로 다룰것 이라고요. 그런데 왜 하필 그 길일까요? 대체 기울기(surrogate gradient)로 역전파해도 성능은 잘 나오는데 말입니다. 그 답을, 뇌가 애초에 어떻게 배우고 왜 그렇게 잘 견디는지에서 찾아봅니다.

### 뇌의 설계도는 '정답'을 저장하지 않는다
사람의 유전자는 약 2만 개이고, 그 중 단백질을 만드는 부분은 전체 DNA의 1~1.5%뿐입니다. 이 정보량으로는 약 860억 개 뉴런과 100조 개에 이르는 연결을 하나하나 지정할 수 없습니다. 그래서 유전체는 담는 것은 완성된 배선도가 아니라 대략의 **① 구조(아키텍처)**, **② 국소 학습 규칙**, **③ 발달 순서**, **④ 안정화 장치** 정도입니다. 나머지 배선은 태어난 뒤 경험이 채웁니다. 즉 뇌의 견고함(robustness)은 "미리 다 알고 있어서"가 아니라, 흔들려도 스스로 되잡는 **동적 자기수정**에서 나옵니다.

### 자기수정을 떠받치는 원리 — 사실 딥러닝이 이미 흉내 내고 있다
이 자기수정을 만드는 생물학의 장치 몇 가지는, 우리가 딥러닝에서 이미 쓰는 기법과 닮아 있습니다.

| 생물학 원리 | 무엇인가 | 딥러닝에서 닮은 것 |
|---|---|---|
| **Canalization**(발생적 완충) | 작은 교란에도 최종 형태가 골짜기로 유도되어 흔들리지 않음(Waddington의 발생 지형) | 넓은 골짜기의 **평평한 최소점(flat minima)** — 파라미터가 조금 어긋나도 성능이 유지됨 |
| **Degeneracy**(축퇴성) | 구조가 다른 부품들이 같은 기능을 대신 수행 → 하나가 고장 나도 안 무너짐 | **드롭아웃(dropout)** 이 강제하는 중복성 |
| **Modularity**(모듈성) | 시각·청각·언어처럼 기능이 나뉘어 한 곳의 손상이 전체로 번지지 않음 | **MoE(전문가 혼합)** 구조 |
| **Homeostatic plasticity**(항상성 가소성) | 빠른 국소 학습 위에 느린 전역 정규화를 얹어 발화율·흥분/억제 균형을 지킴 | **정규화 + 학습률 스케줄링** (그리고 곧 02편에서 쓸 **가중치 정규화**!) |

여기서 핵심은 **② 국소 학습 규칙**입니다. 실제 시냅스는 멀리 떨어진 전역 오차를 역방향으로 받아 바뀌지 않습니다. 자기 양끝 두 뉴런의 활동, 발화 타이밍, 그리고 도파민 같은 조절 신호 — 이 **국소 정보만으로** 바뀝니다. 다음 편부터 만들 STDP(Ch02), 3인자 학습(Ch03)이 그 방법입니다. 대체 기울기+역전파는 강력하지만, 순전파와 역전파가 서로 다른 계산을 쓰고, 전역 정보를 요구한다는 점에서 ②의 방법과는 거리가 있습니다.

### 그래서 이 시리즈의 노선
생물학적 학습을 따라가는 이유는 '더 뇌를 닮아서'라는 낭만만이 아닙니다. **국소·온라인·자기수정**이라는 성질이 (a) 뉴로모픽 칩에서 값싸게 돌고, (b) 시간 전체를 저장하지 않아 메모리를 아끼며, (c) 그 자체로 견고함을 만들기 때문입니다. 물론 대가도 있습니다 — 현재 순수 국소 규칙은 대체 기울기+역전파만큼의 성능에는 대체로 못 미칩니다. **정확도(역전파) ↔ 생물학적 타당성·효율(국소 규칙)** 사이의 이 긴장이 시리즈 전체를 관통하며, 02편의 STDP가 그 첫 실마리입니다.

> 각주 — *canalization*: C. H. Waddington의 발생 지형(epigenetic landscape) 개념. *degeneracy*: Edelman & Gally, "Degeneracy and complexity in biological systems", PNAS 2001.

## 한 걸음 더: 2차(2-state) 뉴런  *(선택)*

*본문 흐름과 독립적인 곁가지입니다 — 건너뛰고 02편으로 가도 됩니다. 이 시리즈의 나머지 편은 계속 1차 `snn.Leaky`를 쓰지만, 방금 만든 뉴런을 한 겹 더 현실적으로 바꾸면 어떻게 되는지 궁금하다면 읽어 보세요.*

1절의 LIF에서는 입력 스파이크가 막전위에 **즉시** 더해졌습니다(그 순간 훅 올라감). 그런데 실제 시냅스에서는 앞 뉴런이 스파이크를 내면 신경전달물질이 퍼지며 뒤 뉴런의 전류가 **몇 스텝에 걸쳐 서서히 올랐다가 잦아듭니다.** 이 부드러운 시냅스 후 전위를 **EPSP(흥분성 시냅스 후 전위, excitatory postsynaptic potential)** 라 합니다. 이 모양을 담으려면 상태 변수를 하나 더 둡니다.

- **1차(1-state) LIF**: 상태가 막전위 $V$ 하나. 입력이 곧장 $V$ 로.
- **2차(2-state)**: 상태가 **시냅스 전류 $I_{syn}$** 와 막전위 $V$ 둘. 입력이 먼저 $I_{syn}$ 로 들어가고, $I_{syn}$ 이 자체 감쇠하며 $V$ 로 흘러듭니다.

그래서 감쇠 계수도 두 개입니다: $\alpha$ 는 시냅스 전류의 누수, $\beta$ 는 (1절 그대로) 막전위의 누수.

$$ I_{syn}[t] = \alpha\,I_{syn}[t-1] + w\,S_{in}[t], \qquad V[t] = \beta\,V[t-1] + I_{syn}[t] - S[t-1]\,V_{th} $$

입력 스파이크 한 발이 $I_{syn}$ 에 얹혀 $\alpha$ 로 천천히 새면서 $V$ 로 흘러 들어가므로, $V$ 는 즉각 점프 대신 **완만한 봉우리**를 그립니다. snntorch에서는 `snn.Synaptic(alpha, beta)` 가 이 뉴런이고, 상태가 둘이라 호출·반환도 하나씩 늘어납니다.

```python
spk, syn, mem = neuron(input, syn, mem)   # (입력, 직전 시냅스전류, 직전 막전위) -> (스파이크, 갱신된 둘)
```

아래에서 **같은 단일 입력 스파이크**(t=5에 한 발)에 1차 `snn.Leaky`와 2차 `snn.Synaptic`을 나란히 굴려, 막전위 모양이 어떻게 갈리는지 봅니다. (`snn.Alpha` 를 쓰면 상승·하강이 더 대칭적인 alpha-함수 시냅스가 되는데, 원리는 같습니다.)

In [ ]:
# 2차 뉴런: 입력 스파이크 한 발에 대한 막전위 반응 비교 (1차 vs 2차)
T = 60
spk_in = torch.zeros(T); spk_in[5] = 1.0     # t=5에 입력 스파이크 딱 한 발
w = 0.5                                        # 시냅스 가중치

# 막전위 '모양'만 보려고 임계값을 아주 크게 잡아 발화(리셋)를 막는다
leaky = snn.Leaky(beta=0.9, threshold=1e9)                 # 1차: 상태 = 막전위 하나
synap = snn.Synaptic(alpha=0.9, beta=0.8, threshold=1e9)   # 2차: 상태 = 시냅스전류 + 막전위

mem_l = torch.zeros(1)                          # 1차 막전위
syn_s = torch.zeros(1); mem_s = torch.zeros(1)  # 2차 시냅스전류, 막전위
V_leaky, V_syn = [], []
for t in range(T):
    x = (w * spk_in[t]).reshape(1)
    _, mem_l = leaky(x, mem_l)                  # 1차: 반환 2개 (스파이크, 막전위)
    _, syn_s, mem_s = synap(x, syn_s, mem_s)    # 2차: 반환 3개 (스파이크, 시냅스전류, 막전위)
    V_leaky.append(mem_l.item()); V_syn.append(mem_s.item())

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(V_leaky, marker=".", ms=3, label="1차 snn.Leaky — 즉각 점프 후 감쇠")
ax.plot(V_syn,   marker=".", ms=3, label="2차 snn.Synaptic — 완만한 EPSP 봉우리")
ax.axvline(5, color="gray", ls=":", lw=1); ax.text(5.5, ax.get_ylim()[1]*0.9, "입력 t=5", fontsize=8)
ax.set_xlabel("시간 스텝 t"); ax.set_ylabel("막전위 V[t]")
ax.set_title("입력 스파이크 한 발에 대한 막전위: 1차 vs 2차")
ax.legend(); plt.tight_layout(); plt.show()

print(f"1차: t=5에 즉시 {max(V_leaky):.2f}로 올랐다가 바로 감쇠")
print(f"2차: t={int(np.argmax(V_syn))} 부근까지 서서히 올라 봉우리 {max(V_syn):.2f} (EPSP 모양)")